In [1]:
"""
================================================================================
  CELL 0 — RUN THIS CELL ALONE FIRST, THEN RESTART RUNTIME, THEN RUN CELL 1
================================================================================
Paste and run only the block below in its own Colab cell:



Then: Runtime → Restart Runtime → run Cell 1 (everything below).
================================================================================

================================================================================
  Multi-Horizon Temporal Fusion Transformer (TFT) — KBS Q1 Journal Submission
  Target Journal : Knowledge-Based Systems (Q1, Elsevier)
  Namespace fix  : uses `lightning.pytorch` throughout (NOT `pytorch_lightning`)
  Tested on      : Colab T4, pytorch-forecasting>=1.1.1, lightning>=2.1
================================================================================
"""
!pip install pytorch-forecasting --quiet
!pip install lightning --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.8/399.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 66.0 MB/s eta 0:00:00


In [5]:
"""
================================================================================
  Multi-Horizon Temporal Fusion Transformer (TFT) — KBS Q1 Journal Submission
  Target Journal  : Knowledge-Based Systems (Q1, Elsevier)
  Environment     : Google Colab T4
  Versions tested : pytorch-forecasting==1.6.1, lightning==2.6.1,
                    torch==2.10.0+cu128
  Install cell (run once, then Runtime → Restart Runtime):
      !pip install pytorch-forecasting lightning --quiet
================================================================================
"""

# ══════════════════════════════════════════════════════════════════════════════
#  0. IMPORTS
# ══════════════════════════════════════════════════════════════════════════════
import os, gc, time, warnings, logging
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import lightning.pytorch as pl                          # ← correct namespace
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_forecasting.data import GroupNormalizer

from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import pytorch_forecasting as _pf

warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.WARNING)
logging.getLogger("pytorch_forecasting").setLevel(logging.WARNING)

# ── Version banner ─────────────────────────────────────────────────────────────
print(f"  lightning            : {pl.__version__}")
print(f"  pytorch-forecasting  : {_pf.__version__}")
print(f"  torch                : {torch.__version__}")
print(f"  CUDA                 : {torch.cuda.is_available()}")

assert issubclass(TemporalFusionTransformer, pl.LightningModule), (
    "Namespace mismatch — TFT does not inherit from lightning.pytorch.LightningModule.\n"
    "Run:  !pip install --force-reinstall pytorch-forecasting lightning --quiet\n"
    "Then restart the runtime."
)
print("  [OK] TFT ↔ LightningModule namespace aligned.\n")

pl.seed_everything(42, workers=True)

# ══════════════════════════════════════════════════════════════════════════════
#  1. CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
INPUT_DIR  = "/content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/"
OUTPUT_DIR = "/content/drive/MyDrive/KBS_Paper/Outputs/8_TFT_KBS/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

HORIZONS          = [3, 6, 12, 24]
STEPS_PER_HORIZON = {h: h // 3 for h in HORIZONS}
TARGET_COL        = "target_buoy_hs"
GROUP_COL         = "group_id"
TIME_IDX_COL      = "time_idx"
TOP_K             = 40
MAX_ENCODER_LEN   = 8          # 8 × 3 h = 24 h history
BATCH_SIZE        = 64
NUM_WORKERS       = 2
MAX_EPOCHS        = 60
PATIENCE          = 4
VIS_HORIZON       = 6          # horizon used for Q1 XAI figures

CANDIDATE_CYCLICAL = [          # adjust to your actual column names
    "hour_sin", "hour_cos",
    "day_sin",  "day_cos",
    "month_sin","month_cos",
    "doy_sin",  "doy_cos",
]

# ══════════════════════════════════════════════════════════════════════════════
#  2. BFloat16 ATTENTION MASK PATCH
#     Must happen BEFORE any TFT is instantiated.
#     pytorch-forecasting >= 1.4 uses mask_bias = -1e9 which overflows float16.
#     bfloat16 has fp32 range so technically safe, but the patch is defensive.
# ══════════════════════════════════════════════════════════════════════════════
try:
    import pytorch_forecasting.models.temporal_fusion_transformer.sub_modules as _sm
    _orig_init = _sm.ScaledDotProductAttention.__init__
    def _patched_init(self, *args, **kwargs):
        _orig_init(self, *args, **kwargs)
        self.mask_bias = -1e4    # safe for both fp16 and bf16
    _sm.ScaledDotProductAttention.__init__ = _patched_init
    print("  [OK] Attention mask_bias patched to -1e4 (AMP-safe)")
except Exception as _patch_err:
    print(f"  [WARN] Attention patch skipped: {_patch_err}")

# ══════════════════════════════════════════════════════════════════════════════
#  3. DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("  STEP 1 — Loading datasets")
print("=" * 70)

df_train_raw = pd.read_csv(os.path.join(INPUT_DIR, "X_train_KBS.csv"))
df_oos_raw   = pd.read_csv(os.path.join(INPUT_DIR, "X_oos_KBS.csv"))

# Drop any residual unnamed index columns
for _df in [df_train_raw, df_oos_raw]:
    _df.drop(columns=[c for c in _df.columns if c.startswith("Unnamed")],
             inplace=True)

print(f"  Train : {df_train_raw.shape}    OOS : {df_oos_raw.shape}")

# Preserve datetime if present
_has_time = "time" in df_train_raw.columns
if _has_time:
    _train_time = pd.to_datetime(df_train_raw["time"]).reset_index(drop=True)
    _oos_time   = pd.to_datetime(df_oos_raw["time"]).reset_index(drop=True)
else:
    _train_time = pd.Series(range(len(df_train_raw)))
    _oos_time   = pd.Series(range(len(df_oos_raw)))

# ══════════════════════════════════════════════════════════════════════════════
#  4. FEATURE IDENTIFICATION
# ══════════════════════════════════════════════════════════════════════════════
_meta       = [c for c in ["time", TARGET_COL] if c in df_train_raw.columns]
_feat_cols  = [c for c in df_train_raw.columns if c not in _meta]
CYCLICAL_COLS  = [c for c in CANDIDATE_CYCLICAL if c in _feat_cols]
CANDIDATE_UNK  = [c for c in _feat_cols if c not in CYCLICAL_COLS]

print(f"  Feature columns     : {len(_feat_cols)}")
print(f"  Cyclical time feats : {len(CYCLICAL_COLS)} → {CYCLICAL_COLS}")
print(f"  Candidate unknown   : {len(CANDIDATE_UNK)}")

# ══════════════════════════════════════════════════════════════════════════════
#  5. GLOBAL SelectKBest
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n  STEP 2 — SelectKBest (f_regression, k={TOP_K})")
_X = df_train_raw[CANDIDATE_UNK].fillna(0).values
_y = df_train_raw[TARGET_COL].fillna(0).values
_sel = SelectKBest(f_regression, k=min(TOP_K, len(CANDIDATE_UNK)))
_sel.fit(_X, _y)
UNKNOWN_REALS = [c for c, m in zip(CANDIDATE_UNK, _sel.get_support()) if m]
KNOWN_REALS   = [TIME_IDX_COL] + CYCLICAL_COLS
print(f"  Selected {len(UNKNOWN_REALS)} unknown_reals")
del _X, _y, _sel; gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
#  6. DATASET UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def _add_meta(df: pd.DataFrame) -> pd.DataFrame:
    """Add time_idx and group_id columns required by TimeSeriesDataSet."""
    df = df.copy().reset_index(drop=True)
    df[TIME_IDX_COL] = np.arange(len(df), dtype=np.int32)
    df[GROUP_COL]    = "buoy"
    return df


def _shift_target(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    """Shift target, drop NaN rows, re-create monotonic time_idx."""
    d = df.copy()
    d["__y"] = d[TARGET_COL].shift(-steps)
    d.dropna(subset=["__y"], inplace=True)
    d.reset_index(drop=True, inplace=True)
    d[TIME_IDX_COL] = np.arange(len(d), dtype=np.int32)
    return d


def build_train_val(df: pd.DataFrame, steps: int):
    d        = _shift_target(df, steps)
    n_val    = max(int(len(d) * 0.15), MAX_ENCODER_LEN + 2)
    cutoff   = int(d[TIME_IDX_COL].iloc[len(d) - n_val])
    known_r  = [c for c in KNOWN_REALS  if c in d.columns]
    unk_r    = [c for c in UNKNOWN_REALS if c in d.columns] + [TARGET_COL]

    train_ds = TimeSeriesDataSet(
        d,
        time_idx                   = TIME_IDX_COL,
        target                     = "__y",
        group_ids                  = [GROUP_COL],
        min_encoder_length         = MAX_ENCODER_LEN // 2,
        max_encoder_length         = MAX_ENCODER_LEN,
        min_prediction_length      = 1,
        max_prediction_length      = 1,
        static_categoricals        = [GROUP_COL],
        time_varying_known_reals   = known_r,
        time_varying_unknown_reals = unk_r,
        target_normalizer          = GroupNormalizer(
            groups=[GROUP_COL], transformation="softplus"
        ),
        add_relative_time_idx  = True,
        add_target_scales      = True,
        add_encoder_length     = True,
        allow_missing_timesteps= False,
        predict_mode           = False,
    )
    val_ds = TimeSeriesDataSet.from_dataset(
        train_ds, d,
        min_prediction_idx = cutoff,
        predict            = False,
        stop_randomization = True,
    )
    return train_ds, val_ds, d


def build_oos(df: pd.DataFrame, steps: int, train_ds: TimeSeriesDataSet):
    d = _shift_target(df, steps)
    oos_ds = TimeSeriesDataSet.from_dataset(
        train_ds, d, predict=False, stop_randomization=True
    )
    return oos_ds, d


def make_loader(ds: TimeSeriesDataSet, shuffle: bool):
    return ds.to_dataloader(
        train=shuffle, batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

# ══════════════════════════════════════════════════════════════════════════════
#  7. PREDICTION EXTRACTION
#     pytorch-forecasting 1.6.x predict() with mode="prediction" returns a
#     plain Tensor (median quantile), NOT a named-tuple.
#     With return_y=True and return_index=True it returns a namedtuple-like
#     object. We handle both cases.
# ══════════════════════════════════════════════════════════════════════════════

def run_predict(model, loader):
    """
    Run inference using model.predict().
    Returns (predictions_np, actuals_np, index_df).
    """
    raw = model.predict(
        loader,
        mode         = "prediction",    # returns median quantile point forecast
        return_y     = True,
        return_index = True,
    )

    # PF 1.6.x: raw is an OutputMixIn with attributes output, y, index
    if hasattr(raw, "output"):
        preds  = raw.output
        actual = raw.y[0]
        idx_df = raw.index
    else:
        # Fallback: raw is a plain tensor
        preds  = raw
        actual = None
        idx_df = None

    # preds shape: (N,) or (N, Q) — take median column if 2-D
    preds = preds.cpu().float()
    if preds.ndim == 2:
        preds = preds[:, preds.shape[1] // 2]
    preds = preds.numpy().flatten()

    if actual is not None:
        actual = actual.cpu().float().numpy().flatten()
    else:
        actual = np.full(len(preds), np.nan)

    return preds, actual, idx_df


def build_pred_df(preds, actual, idx_df, ref_df, time_series,
                  split, horizon):
    """Map time_idx back to timestamps and build the DSS output DataFrame."""
    if idx_df is not None and TIME_IDX_COL in idx_df.columns:
        raw_idx     = idx_df[TIME_IDX_COL].values.flatten()
        idx_to_time = dict(zip(ref_df[TIME_IDX_COL].values, time_series.values))
        timestamps  = [idx_to_time.get(i, i) for i in raw_idx]
    else:
        timestamps = list(range(len(preds)))

    n = min(len(timestamps), len(actual), len(preds))
    return pd.DataFrame({
        "time"          : timestamps[:n],
        "split"         : split,
        "horizon_hours" : horizon,
        "actual_hs"     : actual[:n],
        "predicted_hs"  : preds[:n],
    })

# ══════════════════════════════════════════════════════════════════════════════
#  8. METRICS
# ══════════════════════════════════════════════════════════════════════════════

def compute_metrics(actual, predicted, split, horizon):
    mask = actual != 0
    mape = (float(np.mean(np.abs((actual[mask] - predicted[mask])
                                  / actual[mask]))) * 100) if mask.any() else np.nan
    return {
        "horizon_hours": horizon,
        "split"        : split,
        "RMSE"  : round(float(np.sqrt(mean_squared_error(actual, predicted))), 6),
        "MAE"   : round(float(mean_absolute_error(actual, predicted)),          6),
        "R2"    : round(float(r2_score(actual, predicted)),                     6),
        "MAPE_%": round(mape, 4),
    }

# ══════════════════════════════════════════════════════════════════════════════
#  9. PLOTTING
# ══════════════════════════════════════════════════════════════════════════════
PLOT_DPI = 600

def _save(fig, stem):
    for ext in ["png", "tiff"]:
        p = os.path.join(OUTPUT_DIR, f"{stem}.{ext}")
        fig.savefig(p, dpi=PLOT_DPI, bbox_inches="tight", format=ext)
        print(f"    Saved → {p}")
    plt.close(fig)


def plot_density_scatter(actual, predicted, horizon):
    sns.set_style("ticks")
    fig, ax = plt.subplots(figsize=(6, 6))
    h2 = ax.hist2d(actual.astype(float), predicted.astype(float),
                   bins=80, cmap="viridis", density=True)
    plt.colorbar(h2[3], ax=ax, label="Density")
    lo, hi = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="1:1 line")
    ax.set_xlabel(r"Observed $H_s$ (m)",  fontsize=13)
    ax.set_ylabel(r"Predicted $H_s$ (m)", fontsize=13)
    ax.set_title(f"TFT Density Scatter | +{horizon}h | OOS", fontsize=14)
    ax.legend(fontsize=11); sns.despine(fig=fig)
    _save(fig, f"fig1_density_scatter_{horizon}h_oos")


def plot_storm_window(actual, predicted, time_vec, horizon):
    sns.set_style("ticks")
    peak = int(np.argmax(actual))
    half = int(14 * 24 / 3) // 2
    i0, i1 = max(0, peak - half), min(len(actual), peak + half)
    t_sl = time_vec.reset_index(drop=True).iloc[i0:i1]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t_sl, actual[i0:i1],    lw=1.5, color="#1f77b4", label=r"Observed $H_s$")
    ax.plot(t_sl, predicted[i0:i1], lw=1.5, color="#d62728",
            linestyle="--", label=f"TFT +{horizon}h")
    ax.axvline(t_sl.iloc[peak - i0], color="grey", lw=0.8,
               linestyle=":", label="Storm peak")
    ax.set_xlabel("Time (UTC)", fontsize=13)
    ax.set_ylabel(r"$H_s$ (m)", fontsize=13)
    ax.set_title(f"TFT 14-Day Storm Window | +{horizon}h | OOS", fontsize=14)
    ax.legend(fontsize=10); sns.despine(fig=fig)
    _save(fig, f"fig2_storm_window_{horizon}h_oos")


def plot_variable_importance(interp, horizon):
    sns.set_style("ticks")
    enc = interp.get("encoder_variables", pd.Series(dtype=float))
    dec = interp.get("decoder_variables", pd.Series(dtype=float))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    for ax, data, title in zip(
        axes, [enc.sort_values(ascending=False).head(15),
               dec.sort_values(ascending=False).head(15)],
        ["Encoder Variable Importance", "Decoder Variable Importance"],
    ):
        if data.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes)
        else:
            colors = sns.color_palette("viridis_r", len(data))
            ax.barh(data.index[::-1], data.values[::-1], color=colors[::-1])
            ax.set_xlabel("Importance Score", fontsize=12)
        ax.set_title(title, fontsize=13); sns.despine(ax=ax)
    fig.suptitle(f"TFT Variable Importance | +{horizon}h | OOS",
                 fontsize=14, y=1.01)
    _save(fig, f"fig3_tft_variable_importance_{horizon}h")


def plot_attention(interp, horizon):
    att = interp.get("attention", None)
    if att is None:
        print("    [WARN] No attention key — skipping attention plot.")
        return
    if isinstance(att, torch.Tensor):
        att = att.cpu().float().numpy()
    att = np.array(att, dtype=float)
    if att.ndim > 1:
        att = att.mean(axis=tuple(range(att.ndim - 1)))   # reduce all but last
    sns.set_style("ticks")
    fig, ax = plt.subplots(figsize=(8, 4))
    pal = sns.color_palette("viridis", len(att))
    ax.bar(range(len(att)), att, color=pal)
    ax.invert_xaxis()
    ax.set_xlabel("Encoder Time Step (← past)", fontsize=12)
    ax.set_ylabel("Mean Attention Weight",       fontsize=12)
    ax.set_title(f"TFT Attention | +{horizon}h | OOS", fontsize=13)
    sns.despine(fig=fig)
    _save(fig, f"fig4_tft_attention_{horizon}h")

# ══════════════════════════════════════════════════════════════════════════════
#  10. PRE-PROCESS META COLUMNS
# ══════════════════════════════════════════════════════════════════════════════
df_train_meta = _add_meta(df_train_raw)
df_oos_meta   = _add_meta(df_oos_raw)

# ══════════════════════════════════════════════════════════════════════════════
#  11. MAIN TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════
all_metrics    = []
all_train_preds= []
all_oos_preds  = []

for horizon in HORIZONS:
    h_steps = STEPS_PER_HORIZON[horizon]
    print("\n" + "=" * 70)
    print(f"  HORIZON: +{horizon}h  ({h_steps} step(s) at 3-h resolution)")
    print("=" * 70)
    t0 = time.time()

    # ── 11a. Datasets ──────────────────────────────────────────────────────────
    print("  Building TimeSeriesDataSets …")
    train_ds, val_ds, df_tr_h = build_train_val(df_train_meta, h_steps)
    oos_ds,   df_oos_h        = build_oos(df_oos_meta, h_steps, train_ds)

    # Aligned time vectors (after shift+drop, lengths may differ from raw)
    n_tr  = len(df_tr_h)
    n_oos = len(df_oos_h)
    tr_times  = (_train_time.iloc[:n_tr].reset_index(drop=True)
                 if _has_time else pd.Series(range(n_tr)))
    oos_times = (_oos_time.iloc[:n_oos].reset_index(drop=True)
                 if _has_time else pd.Series(range(n_oos)))

    train_loader = make_loader(train_ds, shuffle=True)
    val_loader   = make_loader(val_ds,   shuffle=False)
    oos_loader   = make_loader(oos_ds,   shuffle=False)

    # ── 11b. Model ─────────────────────────────────────────────────────────────
    print("  Instantiating TFT …")
    tft = TemporalFusionTransformer.from_dataset(
        train_ds,
        learning_rate           = 3e-3,
        hidden_size             = 64,
        attention_head_size     = 2,
        dropout                 = 0.15,
        hidden_continuous_size  = 32,
        output_size             = 7,
        loss                    = QuantileLoss(),
        log_interval            = -1,    # ← CRITICAL: disables matplotlib logging
        reduce_on_plateau_patience = 3,
    )
    print(f"  Trainable params: "
          f"{sum(p.numel() for p in tft.parameters() if p.requires_grad):,}")

    # ── 11c. Trainer ───────────────────────────────────────────────────────────
    # REMOVED LearningRateMonitor — it requires a logger and crashes with
    # logger=False. EarlyStopping + ModelCheckpoint are sufficient.
    callbacks = [
        EarlyStopping(
            monitor   = "val_loss",
            patience  = PATIENCE,
            mode      = "min",
            min_delta = 1e-4,
            verbose   = True,
        ),
        ModelCheckpoint(
            monitor   = "val_loss",
            mode      = "min",
            dirpath   = OUTPUT_DIR,
            filename  = f"tft_best_{horizon}h",
            save_top_k= 1,
        ),
    ]

    trainer = Trainer(
        accelerator          = "gpu",
        devices              = 1,
        precision            = "bf16-mixed",   # ← bfloat16: no mask overflow
        gradient_clip_val    = 0.1,
        max_epochs           = MAX_EPOCHS,
        callbacks            = callbacks,
        enable_progress_bar  = True,
        logger               = False,          # ← no logger = no LRMonitor
        enable_model_summary = False,
    )

    # ── 11d. Train ─────────────────────────────────────────────────────────────
    print(f"  Training (max {MAX_EPOCHS} epochs, patience={PATIENCE}) …")
    trainer.fit(tft, train_dataloaders=train_loader,
                val_dataloaders=val_loader)
    print(f"  Done — epoch {trainer.current_epoch} | "
          f"{(time.time()-t0)/60:.1f} min")

    # Reload best weights
    best_path = trainer.checkpoint_callback.best_model_path
    if best_path and os.path.exists(best_path):
        tft = TemporalFusionTransformer.load_from_checkpoint(best_path)
        print(f"  Loaded best: {os.path.basename(best_path)}")
    tft.eval()

    # ── 11e. Inference — TRAIN ─────────────────────────────────────────────────
    print("  Inference → TRAIN …")
    tr_pred_loader = make_loader(train_ds, shuffle=False)
    tr_preds, tr_actual, tr_idx = run_predict(tft, tr_pred_loader)
    tr_df = build_pred_df(tr_preds, tr_actual, tr_idx,
                          df_tr_h, tr_times, "train", horizon)
    all_train_preds.append(tr_df)
    all_metrics.append(compute_metrics(
        tr_df["actual_hs"].values, tr_df["predicted_hs"].values, "train", horizon
    ))

    # ── 11f. Inference — OOS ──────────────────────────────────────────────────
    print("  Inference → OOS …")
    oos_preds, oos_actual, oos_idx = run_predict(tft, oos_loader)
    oos_df = build_pred_df(oos_preds, oos_actual, oos_idx,
                           df_oos_h, oos_times, "oos", horizon)
    all_oos_preds.append(oos_df)
    all_metrics.append(compute_metrics(
        oos_df["actual_hs"].values, oos_df["predicted_hs"].values, "oos", horizon
    ))

    # ── 11g. Q1 XAI Figures (VIS_HORIZON only) ────────────────────────────────
    if horizon == VIS_HORIZON:
        print(f"\n  Generating Q1 XAI figures for +{horizon}h OOS …")
        _a = oos_df["actual_hs"].values.astype(float)
        _p = oos_df["predicted_hs"].values.astype(float)

        try:
            plot_density_scatter(_a, _p, horizon)
        except Exception as e:
            print(f"  [WARN] Density scatter: {e}")

        try:
            plot_storm_window(_a, _p, oos_times, horizon)
        except Exception as e:
            print(f"  [WARN] Storm window: {e}")

        # Native TFT XAI — interpret_output()
        try:
            # mode="raw" returns the full output dict needed by interpret_output
            raw_out = tft.predict(
                oos_loader,
                mode   = "raw",
                return_y = False,
                return_index = False,
            )
            interp = tft.interpret_output(raw_out, reduction="sum")
            plot_variable_importance(interp, horizon)
            plot_attention(interp, horizon)
            print("  XAI figures saved.")
        except Exception as e:
            print(f"  [WARN] XAI interpretation failed: {e}")
            print("  Skipping interpretation figures — training results intact.")

    # ── 11h. Cleanup ───────────────────────────────────────────────────────────
    del tft, trainer
    del train_ds, val_ds, oos_ds
    del train_loader, val_loader, oos_loader, tr_pred_loader
    torch.cuda.empty_cache(); gc.collect()
    print(f"  VRAM cleared. Wall-time: {(time.time()-t0)/60:.1f} min")

# ══════════════════════════════════════════════════════════════════════════════
#  12. EXPORT DSS ARTEFACTS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  STEP 3 — Exporting DSS artefacts")
print("=" * 70)

EXPORT_COLS = ["time", "split", "horizon_hours", "actual_hs", "predicted_hs"]

# Metrics
metrics_df = pd.DataFrame(all_metrics)[
    ["horizon_hours", "split", "RMSE", "MAE", "R2", "MAPE_%"]
]
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "tft_metrics_summary.csv"), index=False)
print("\n  tft_metrics_summary.csv")
print(metrics_df.to_string(index=False))

# Train predictions
tr_out = pd.concat(all_train_preds, ignore_index=True)[EXPORT_COLS]
tr_out.to_csv(os.path.join(OUTPUT_DIR, "tft_train_predictions.csv"), index=False)
print(f"\n  tft_train_predictions.csv  ({len(tr_out):,} rows)")

# OOS predictions
oos_out = pd.concat(all_oos_preds, ignore_index=True)[EXPORT_COLS]
oos_out.to_csv(os.path.join(OUTPUT_DIR, "tft_oos_predictions.csv"), index=False)
print(f"  tft_oos_predictions.csv    ({len(oos_out):,} rows)")

# ══════════════════════════════════════════════════════════════════════════════
#  13. FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  FINAL PERFORMANCE SUMMARY")
print("=" * 70)
for _, row in metrics_df.iterrows():
    print(f"  +{int(row.horizon_hours):2d}h | {row.split:5s} | "
          f"RMSE={row.RMSE:.4f}  MAE={row.MAE:.4f}  "
          f"R²={row.R2:.4f}  MAPE={row['MAPE_%']:.2f}%")

print(f"\n  Artefacts → {OUTPUT_DIR}")
print("  ✓ Script completed successfully.")

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


  lightning            : 2.6.1
  pytorch-forecasting  : 1.6.1
  torch                : 2.10.0+cu128
  CUDA                 : True
  [OK] TFT ↔ LightningModule namespace aligned.

  [OK] Attention mask_bias patched to -1e4 (AMP-safe)
  STEP 1 — Loading datasets
  Train : (12477, 723)    OOS : (5605, 723)
  Feature columns     : 721
  Cyclical time feats : 4 → ['hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
  Candidate unknown   : 717

  STEP 2 — SelectKBest (f_regression, k=40)
  Selected 40 unknown_reals

  HORIZON: +3h  (1 step(s) at 3-h resolution)
  Building TimeSeriesDataSets …


INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:L

  Instantiating TFT …
  Trainable params: 668,556
  Training (max 60 epochs, patience=4) …


Output()

...

  Done — epoch 60 | 53.6 min


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

  Loaded best: tft_best_24h.ckpt
  Inference → TRAIN …


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

  Inference → OOS …
  VRAM cleared. Wall-time: 54.1 min

  STEP 3 — Exporting DSS artefacts

  tft_metrics_summary.csv
 horizon_hours split     RMSE      MAE        R2  MAPE_%
             3 train 0.123665 0.049455  0.808456 13.7340
             3   oos 0.244600 0.102196  0.344244 23.5047
             6 train 0.124395 0.050636  0.806204 13.9716
             6   oos 0.264810 0.123919  0.231387 30.8079
            12 train 0.226634 0.112931  0.356840 28.1958
            12   oos 0.261692 0.122119  0.249369 28.5758
            24 train 0.115201 0.052013  0.833868 14.3350
            24   oos 0.308800 0.175169 -0.045058 45.3847

  tft_train_predictions.csv  (49,893 rows)
  tft_oos_predictions.csv    (22,405 rows)

  FINAL PERFORMANCE SUMMARY
  + 3h | train | RMSE=0.1237  MAE=0.0495  R²=0.8085  MAPE=13.73%
  + 3h | oos   | RMSE=0.2446  MAE=0.1022  R²=0.3442  MAPE=23.50%
  + 6h | train | RMSE=0.1244  MAE=0.0506  R²=0.8062  MAPE=13.97%
  + 6h | oos   | RMSE=0.2648  MAE=0.1239  R²=0.2314  MAPE